In [3]:
#Polynomial matrix:

import numpy as np
import mpmath as mp

# Pauli matrices
X = np.array([[0, 1],
              [1, 0]], dtype=complex)
Z = np.array([[1, 0],
              [0, -1]], dtype=complex)
I = np.eye(2, dtype=complex)

# Coefficients from Bessel functions
def C0(t):
    return mp.besselj(0, t) + 2*mp.besselj(2, t) + 2*mp.besselj(4, t)

def C2(t):
    return -4*mp.besselj(2, t) - 16*mp.besselj(4, t)

def C4(t):
    return 16*mp.besselj(4, t)

# Build H
def build_H(J1, J2):
    return J1 * X + J2 * Z

# Polynomial in H
def polynomial_H(t, J1, J2):
    H = build_H(J1, J2)
    H2 = np.matmul(H, H)
    H4 = np.matmul(H2, H2)
    P = C0(t)*I + C2(t)*H2 + C4(t)*H4
    return np.array(P, dtype=complex)

# Example usage
t = 1.6
J1, J2 = 0.25, 0.4
P = polynomial_H(t, J1, J2)
print("Polynomial P(t, J1, J2) =\n", P)

Polynomial P(t, J1, J2) =
 [[0.66914094+0.j 0.        +0.j]
 [0.        +0.j 0.66914094+0.j]]


In [3]:
#Polynomial matrix:

import numpy as np
import mpmath as mp

# Pauli matrices
X = np.array([[0, 1],
              [1, 0]], dtype=complex)
Z = np.array([[1, 0],
              [0, -1]], dtype=complex)
I = np.eye(2, dtype=complex)

# Coefficients from Bessel functions
def C0(t):
    return mp.besselj(0, t) + 2*mp.besselj(2, t) + 2*mp.besselj(4, t)

def C2(t):
    return -4*mp.besselj(2, t) - 16*mp.besselj(4, t)

def C4(t):
    return 16*mp.besselj(4, t)

# Build H
def build_H(J1, J2):
    return J1 * X + J2 * Z

# Polynomial in H
def polynomial_H(t, J1, J2):
    H = build_H(J1, J2)
    H2 = np.matmul(H, H)
    H4 = np.matmul(H2, H2)
    P = C0(t)*I + C2(t)*H2 + C4(t)*H4
    return np.array(P, dtype=complex)

# Example usage
t = 1.6
J1, J2 = 0.25, 0.4
P = polynomial_H(t, J1, J2)
print("Polynomial P(t, J1, J2) =\n", P)

Polynomial P(t, J1, J2) =
 [[0.66914094+0.j 0.        +0.j]
 [0.        +0.j 0.66914094+0.j]]


In [ ]:
#cosine part.

In [1]:
#Apply on [1,0]:


import numpy as np
import mpmath as mp
from scipy.special import jv

# Pauli matrices
X = np.array([[0, 1],
              [1, 0]], dtype=complex)
Z = np.array([[1, 0],
              [0, -1]], dtype=complex)
I = np.eye(2, dtype=complex)

# Coefficients from Bessel functions
def C0(t):
    return mp.besselj(0, t) + 2*mp.besselj(2, t) - 2*mp.besselj(4, t) - 2*mp.besselj(6,t)

def C2(t):
    return -4*mp.besselj(2, t) - 16*mp.besselj(4, t) + 20*mp.besselj(6,t)

def C4(t):
    return 16*mp.besselj(4, t) + 96*mp.besselj(6,t)

def C6(t):
    return -64*mp.besselj(6,t)

# Build H
def build_H(J1, J2):
    total = 2 * J1 + J2
    J1 = J1/total
    J2 = J2/total
    return J1 * np.kron(X,I) + J1 * np.kron(I,X) + J2 * np.kron(Z,Z)

# Polynomial in H
def polynomial_H(t, J1, J2, max_degree):
    H = build_H(J1, J2)
    # H2 = H @ H
    # H4 = H2 @ H2
    # P = C0(t)*np.kron(I,I) + C2(t)*H2 + C4(t)*H4
    coeff_total = np.zeros(2 * max_degree)
    T = np.zeros((2 * max_degree, 2 * max_degree))
    T[0][0] = 1
    T[1][1] = 1
    for i in range(2, len(T)):
        T[i,:] = 2 * np.concatenate((np.zeros(1),T[i-1,0:len(T[0])-1]))
        T[i,:] -= T[i-2,:]
    coeff_total = jv(0, t) * T[0, :]
    for k in range(2, len(T), 2):
        coeff_total += 2 * (-1)**(int(k/2)) * jv(k, t) * T[k, :] 
    temp = np.eye(len(H))
    polynomial = np.zeros((len(H),len(H)), dtype = complex)
    for i in range(len(coeff_total)):
        polynomial += temp * coeff_total[i]
        temp = H @ temp
    return polynomial
   

# Apply polynomial on vector (1,0)
def apply_polynomial(t, J1, J2, max_degree):
    # P = build_H(J1, J2)
    P = polynomial_H(t,J1,J2, max_degree)
    print("resultant matrix =\n", P)

    

    v = np.array([[1],[0],[0],[0]], dtype=complex)
    result = P @ v
    return result

# Example usage
t = 1.2
J1, J2 = 0.25, 0.4
max_degree = 4
result = apply_polynomial(t, J1, J2, max_degree)
print("Resulting vector =\n", result)

zero = np.array([[1],[0]])
one = np.array([[0],[1]])
plus = 1/np.sqrt(2) * np.array([[1],[1]])

measurementState = np.kron(zero, np.kron(zero, plus))

print('Result vector extended =\n', np.round(np.kron(result, measurementState).transpose()[0], 4))



resultant matrix =
 [[ 7.59139171e-01+0.j  2.99311398e-19+0.j  5.98622796e-19+0.j
  -1.01978683e-01+0.j]
 [-2.99311398e-19+0.j  7.59139171e-01+0.j -1.01978683e-01+0.j
  -5.98622796e-19+0.j]
 [-2.99311398e-19+0.j -1.01978683e-01+0.j  7.59139171e-01+0.j
  -5.98622796e-19+0.j]
 [-1.01978683e-01+0.j  0.00000000e+00+0.j  0.00000000e+00+0.j
   7.59139171e-01+0.j]]
Resulting vector =
 [[ 7.59139171e-01+0.j]
 [-2.99311398e-19+0.j]
 [-2.99311398e-19+0.j]
 [-1.01978683e-01+0.j]]
Result vector extended =
 [ 0.5368+0.j  0.5368+0.j  0.    +0.j  0.    +0.j  0.    +0.j  0.    +0.j
  0.    +0.j  0.    +0.j -0.    +0.j -0.    +0.j -0.    +0.j -0.    +0.j
 -0.    +0.j -0.    +0.j -0.    +0.j -0.    +0.j -0.    +0.j -0.    +0.j
 -0.    +0.j -0.    +0.j -0.    +0.j -0.    +0.j -0.    +0.j -0.    +0.j
 -0.0721+0.j -0.0721+0.j -0.    +0.j -0.    +0.j -0.    +0.j -0.    +0.j
 -0.    +0.j -0.    +0.j]


In [ ]:
#sine part

In [3]:
#Apply on [1,0]:


import numpy as np
import mpmath as mp
from scipy.special import jv

# Pauli matrices
X = np.array([[0, 1],
              [1, 0]], dtype=complex)
Z = np.array([[1, 0],
              [0, -1]], dtype=complex)
I = np.eye(2, dtype=complex)

# Coefficients from Bessel functions
def C1(t):
    return 2*mp.besselj(1, t) + 6*mp.besselj(3, t) - 2*mp.besselj(5, t) + 2*mp.besselj(7,t)

def C3(t):
    return -8*mp.besselj(3, t) - 40*mp.besselj(5, t) - 80*mp.besselj(7,t)

def C5(t):
    return 32*mp.besselj(3, t) + 224*mp.besselj(5,t)

def C7(t):
    return 128*mp.besselj(7,t)

# Build H
def build_H(J1, J2):
    total = 2 * J1 + J2
    J1 = J1/total
    J2 = J2/total
    return J1 * np.kron(X,I) + J1 * np.kron(I,X) + J2 * np.kron(Z,Z)

# Polynomial in H
def polynomial_H(t, J1, J2, max_degree):
    H = build_H(J1, J2)
    coeff_total = np.zeros(2 * max_degree)
    T = np.zeros((2 * max_degree, 2 * max_degree))
    T[0][0] = 1
    T[1][1] = 1
    for i in range(2, len(T)):
        T[i,:] = 2 * np.concatenate((np.zeros(1),T[i-1,0:len(T[0])-1]))
        T[i,:] -= T[i-2,:]
    coeff_total = 2 * jv(1, t) * T[1, :]
    for k in range(3, len(T), 2):
    
        coeff_total += 2 * (-1)**int((k-1)/2) * jv(k, t) * T[k, :] 
    
    
    temp = np.eye(len(H))
    polynomial = np.zeros((len(H),len(H)), dtype = complex)
    for i in range(len(coeff_total)):
        polynomial += temp * coeff_total[i]
        temp = H @ temp
    return polynomial
   

# Apply polynomial on vector (1,0)
def apply_polynomial(t, J1, J2, max_degree):
    # P = build_H(J1, J2)
    P = polynomial_H(t,J1,J2, max_degree)
    print("resultant matrix =\n", P)

    

    v = np.array([[1],[0],[0],[0]], dtype=complex)
    result = P @ v
    return result

# Example usage
t = 1.2
J1, J2 = 0.25, 0.4
max_degree = 4
result = apply_polynomial(t, J1, J2, max_degree)
print("Resulting vector =\n", result)

zero = np.array([[1],[0]])
one = np.array([[0],[1]])
plus = 1/np.sqrt(2) * np.array([[1],[1]])

measurementState = np.kron(zero, np.kron(zero, plus))

print('Result vector extended =\n', np.round(np.kron(result, measurementState).transpose()[0], 4))



resultant matrix =
 [[ 0.48963523+0.j  0.29428991+0.j  0.29428991+0.j -0.01877137+0.j]
 [ 0.29428991+0.j -0.48963523+0.j  0.01877137+0.j  0.29428991+0.j]
 [ 0.29428991+0.j  0.01877137+0.j -0.48963523+0.j  0.29428991+0.j]
 [-0.01877137+0.j  0.29428991+0.j  0.29428991+0.j  0.48963523+0.j]]
Resulting vector =
 [[ 0.48963523+0.j]
 [ 0.29428991+0.j]
 [ 0.29428991+0.j]
 [-0.01877137+0.j]]
Result vector extended =
 [ 0.3462+0.j  0.3462+0.j  0.    +0.j  0.    +0.j  0.    +0.j  0.    +0.j
  0.    +0.j  0.    +0.j  0.2081+0.j  0.2081+0.j  0.    +0.j  0.    +0.j
  0.    +0.j  0.    +0.j  0.    +0.j  0.    +0.j  0.2081+0.j  0.2081+0.j
  0.    +0.j  0.    +0.j  0.    +0.j  0.    +0.j  0.    +0.j  0.    +0.j
 -0.0133+0.j -0.0133+0.j -0.    +0.j -0.    +0.j -0.    +0.j -0.    +0.j
 -0.    +0.j -0.    +0.j]


In [ ]:
#Add the two code like a+ib

In [4]:
import numpy as np
from scipy.special import jv

# Pauli matrices
X = np.array([[0, 1],
              [1, 0]], dtype=complex)
Z = np.array([[1, 0],
              [0, -1]], dtype=complex)
I = np.eye(2, dtype=complex)

# Build H
def build_H(J1, J2):
    total = 2 * J1 + J2
    J1 = J1/total
    J2 = J2/total
    return J1 * np.kron(X,I) + J1 * np.kron(I,X) + J2 * np.kron(Z,Z)

# Chebyshev matrix coefficients
def chebyshev_matrix(max_degree):
    T = np.zeros((max_degree, max_degree))
    T[0,0] = 1
    if max_degree > 1:
        T[1,1] = 1
    for i in range(2, max_degree):
        T[i,:] = 2 * np.concatenate((np.zeros(1), T[i-1,0:max_degree-1]))
        T[i,:] -= T[i-2,:]
    return T

# Polynomial in H for even or odd Bessel orders
def polynomial_H(t, J1, J2, max_degree, parity='even'):
    H = build_H(J1, J2)
    T = chebyshev_matrix(2 * max_degree)
    coeff_total = np.zeros(2 * max_degree)
    
    if parity == 'even':
        coeff_total = jv(0, t) * T[0,:]
        for k in range(2, len(T), 2):
            coeff_total += 2 * (-1)**(k//2) * jv(k, t) * T[k,:]
    elif parity == 'odd':
        coeff_total = 2 * jv(1, t) * T[1,:]
        for k in range(3, len(T), 2):
            coeff_total += 2 * (-1)**((k-1)//2) * jv(k, t) * T[k,:]
    else:
        raise ValueError("parity must be 'even' or 'odd'")
    
    temp = np.eye(len(H))
    polynomial = np.zeros((len(H), len(H)), dtype=complex)
    for i in range(len(coeff_total)):
        polynomial += temp * coeff_total[i]
        temp = H @ temp
    return polynomial

# Apply combined polynomial a + i b on vector [1,0,0,0]
def apply_combined_polynomial(t, J1, J2, max_degree):
    P_even = polynomial_H(t, J1, J2, max_degree, parity='even')
    P_odd  = polynomial_H(t, J1, J2, max_degree, parity='odd')
    
    P_combined = P_even + 1j * P_odd
    print("Resultant matrix P_combined =\n", P_combined)
    
    v = np.array([[1],[0],[0],[0]], dtype=complex)
    result = P_combined @ v
    return result

# Example usage
t = 1.2
J1, J2 = 0.25, 0.4
max_degree = 4
result = apply_combined_polynomial(t, J1, J2, max_degree)
print("Resulting vector =\n", result)

# Extend with measurement state
zero = np.array([[1],[0]])
one = np.array([[0],[1]])
plus = 1/np.sqrt(2) * np.array([[1],[1]])
measurementState = np.kron(zero, np.kron(zero, plus))

print('Result vector extended =\n', np.round(np.kron(result, measurementState).transpose()[0], 4))

Resultant matrix P_combined =
 [[ 7.59139171e-01+0.48963523j  2.99311398e-19+0.29428991j
   5.98622796e-19+0.29428991j -1.01978683e-01-0.01877137j]
 [-2.99311398e-19+0.29428991j  7.59139171e-01-0.48963523j
  -1.01978683e-01+0.01877137j -5.98622796e-19+0.29428991j]
 [-2.99311398e-19+0.29428991j -1.01978683e-01+0.01877137j
   7.59139171e-01-0.48963523j -5.98622796e-19+0.29428991j]
 [-1.01978683e-01-0.01877137j  0.00000000e+00+0.29428991j
   0.00000000e+00+0.29428991j  7.59139171e-01+0.48963523j]]
Resulting vector =
 [[ 7.59139171e-01+0.48963523j]
 [-2.99311398e-19+0.29428991j]
 [-2.99311398e-19+0.29428991j]
 [-1.01978683e-01-0.01877137j]]
Result vector extended =
 [ 0.5368+0.3462j  0.5368+0.3462j  0.    +0.j      0.    +0.j
  0.    +0.j      0.    +0.j      0.    +0.j      0.    +0.j
 -0.    +0.2081j -0.    +0.2081j -0.    +0.j     -0.    +0.j
 -0.    +0.j     -0.    +0.j     -0.    +0.j     -0.    +0.j
 -0.    +0.2081j -0.    +0.2081j -0.    +0.j     -0.    +0.j
 -0.    +0.j     -0.    

In [1]:
import numpy as np
import mpmath as mp

# Pauli matrices
X = np.array([[0,1],[1,0]], dtype=complex)
Z = np.array([[1,0],[0,-1]], dtype=complex)
I = np.eye(2, dtype=complex)

# Bessel-based coefficients
def C0(t):
    return float(mp.besselj(0, t) + 2*mp.besselj(2, t) - 2*mp.besselj(4, t))
def C2(t):
    return float(-4*mp.besselj(2, t) - 16*mp.besselj(4, t))
def C4(t):
    return float(16*mp.besselj(4, t))

# Build H and polynomial P
def build_H(J1, J2):
    return J1*X + J2*Z

def polynomial_P(t, J1, J2):
    H = build_H(J1, J2)
    H2 = H @ H
    H4 = H2 @ H2
    P = C0(t)*I + C2(t)*H2 + C4(t)*H4
    return np.array(P, dtype=complex)

# basis states
zero_state = np.array([1,0], dtype=complex)
plus_state = np.array([1,1], dtype=complex)/np.sqrt(2)

# Apply to first qubit and build n-qubit state
def apply_to_first_qubit_and_build_full(t, J1, J2, n_qubits):
    P = polynomial_P(t, J1, J2)
    single_out = P @ zero_state
    single_out = single_out / np.linalg.norm(single_out)
    full_state = single_out.copy()
    for _ in range(n_qubits-1):
        full_state = np.kron(full_state, zero_state)
    return full_state

# Probability of measuring |+ 0 0 ... 0>
def probability_first_plus_rest_zero(t, J1, J2, n_qubits):
    full = apply_to_first_qubit_and_build_full(t, J1, J2, n_qubits)
    target = plus_state.copy()
    for _ in range(n_qubits-1):
        target = np.kron(target, zero_state)
    full = full / np.linalg.norm(full)
    target = target / np.linalg.norm(target)
    amplitude = np.vdot(target, full)
    prob = abs(amplitude)**2
    return prob, amplitude, full, target

# Example
t = 1.2
J1 = 0.25
J2 = 0.4
n_qubits = 3

prob, amp, full_state, target_state = probability_first_plus_rest_zero(t, J1, J2, n_qubits)
print("Probability =", prob)
print("Amplitude =", amp)

Probability = 0.5000000000000001
Amplitude = (0.7071067811865476+0j)
